In [ ]:
library(Seurat)
library(Matrix)
library(variancePartition)
library(BiocParallel)
library(limma)
library(ggplot2)
library(dplyr)
library(DESeq2)
library(edgeR)

In [ ]:
library(yaml)

# ── Load pipeline paths from config.yaml (searches up from the working dir) ──
# Single source of truth for file locations; edit config.yaml at the repo root.
find_config <- function(name = "config.yaml", start = getwd()) {
  d <- normalizePath(start, winslash = "/", mustWork = FALSE)
  repeat {
    p <- file.path(d, name)
    if (file.exists(p)) return(p)
    parent <- dirname(d)
    if (identical(parent, d)) stop(paste0(name, " not found in ", start, " or any parent dir"))
    d <- parent
  }
}
cfg_all <- yaml::read_yaml(find_config())

# This pipeline runs over two frameworks (datasets): "genotype" and "fmt". Pick
# which one to run here; cfg_paths becomes that framework's block, so the
# cfg_paths$composition / $outputs references below stay unchanged.
FRAMEWORK <- ""   # e.g. "genotype" or "fmt"; "" -> DE_FRAMEWORK env / active_framework
if (!is.null(cfg_all$frameworks)) {
  fw_name <- if (nzchar(FRAMEWORK)) FRAMEWORK else
             if (nzchar(Sys.getenv("DE_FRAMEWORK"))) Sys.getenv("DE_FRAMEWORK") else
             cfg_all$active_framework
  if (is.null(cfg_all$frameworks[[fw_name]]))
    stop(sprintf("Framework '%s' not in config. Available: %s",
                 fw_name, paste(names(cfg_all$frameworks), collapse = ", ")))
  message("Framework: ", fw_name)
  cfg_paths <- cfg_all$frameworks[[fw_name]]
} else {
  cfg_paths <- cfg_all   # flat/legacy config
}

COMP <- cfg_paths$composition   # composition-engineering + DE iteration design


In [ ]:
make_cfg <- function(root_dir,
                     seurat_rds_name    = "seurat_object.rds",
                     raw_counts_name    = "raw_counts.csv",
                     logged_counts_name = "normalized_counts.csv",
                     features_name      = "features_counts.csv",
                     metadata_name      = "cell_metadata.csv",
                     coords_name        = "coords_xy.csv",
                     quint_labels_name  = "Color_key.xlsm") {
  list(
    root_dir      = root_dir,
    seurat_rds    = file.path(root_dir, seurat_rds_name),
    raw_counts    = file.path(root_dir, raw_counts_name),
    logged_counts = file.path(root_dir, logged_counts_name),
    features      = file.path(root_dir, features_name),
    metadata      = file.path(root_dir, metadata_name),
    coords        = file.path(root_dir, coords_name),
    quint_labels  = file.path(root_dir, quint_labels_name)
  )
}

build_seurat_from_folder <- function(cfg, assay_name = "RNA") {
  message(paste0("Loading from: ", cfg$root_dir))
  counts        <- read.csv(cfg$raw_counts,     row.names = 1, check.names = FALSE)
  logged_counts <- read.csv(cfg$logged_counts,  row.names = 1, check.names = FALSE)
  features      <- read.csv(cfg$features,       row.names = 1, check.names = FALSE)
  meta          <- read.csv(cfg$metadata,       row.names = 1, check.names = FALSE)
  coords        <- read.csv(cfg$coords,         row.names = 1, check.names = FALSE)
  if (ncol(counts) != nrow(features))
    stop(paste("Mismatch: Counts matrix has", ncol(counts), "genes, but features file has", nrow(features)))
  colnames(counts)        <- rownames(features)
  colnames(logged_counts) <- rownames(features)
  common_cells <- Reduce(intersect, list(rownames(counts), rownames(meta),
                                         rownames(coords), rownames(logged_counts)))
  if (length(common_cells) == 0)
    stop("No common Cell IDs found. Check your CSV row names.")
  message(paste0("  Matched ", length(common_cells), " cells across all files."))
  counts        <- counts[common_cells, ]
  logged_counts <- logged_counts[common_cells, ]
  meta          <- meta[common_cells, ]
  coords        <- coords[common_cells, ]
  seu <- CreateSeuratObject(counts = t(counts), meta.data = meta, assay = assay_name)
  seu <- SetAssayData(seu, layer = "data", new.data = as.matrix(t(logged_counts)))
  seu <- AddMetaData(seu, metadata = coords)
  return(seu)
}

# -- Load the single full-cohort export --------------------------------------
# Composition_Engineering.ipynb exports one object carrying per-cell boolean
# masks G1_{seed}_{dev} / G2_{seed}_{dev} (cortex-up / cortex-down) for every
# (deviation magnitude, iteration) pair. Load it once here; the iteration loop
# in the driver cell reconstructs the engineered up/down subsets on demand.
print("Loading full cohort")
cfg_file_base <- make_cfg(cfg_paths$outputs$de_export_base)
full <- build_seurat_from_folder(cfg_file_base)


In [ ]:
# run_de_dream: region-aware Dream LMM
run_de_dream <- function(seurat_obj, dataset_name, out_prefix, save_dir,
                         region_col = "napari_region") {
  message(paste0("\n>>> [DREAM] STARTING FOR: ", dataset_name))
  num_cts  <- length(unique(seurat_obj$cell_type))
  form_str <- if (num_cts > 1) {
    message(paste0("    Detected ", num_cts, " cell types. Including (1|cell_type)."))
    paste0("~ Treatment + log_depth + (1|", region_col, ") + (1|cell_type) + (1|sample_ID)")
  } else {
    message("    Single cell type — removing (1|cell_type) from formula.")
    paste0("~ Treatment + log_depth + (1|", region_col, ") + (1|sample_ID)")
  }
  form_de <- as.formula(form_str)
  message(paste("    Formula:", deparse(form_de)))
  counts <- as.matrix(GetAssayData(seurat_obj, layer = "counts"))
  info   <- seurat_obj@meta.data
  counts <- counts[rowSums(counts) > 0, ]
  dge    <- calcNormFactors(DGEList(counts))
  message("    Calculating voom weights...")
  vobj <- voomWithDreamWeights(dge, form_de, info, BPPARAM = param)
  fit  <- eBayes(dream(vobj, form_de, info, BPPARAM = param))
  target_coef <- grep("Treatment", colnames(fit$coefficients), value = TRUE)
  target_coef <- target_coef[length(target_coef)]
  if (length(target_coef) > 0) {
    de_res       <- topTable(fit, coef = target_coef, number = Inf, sort.by = "P", confint = TRUE)
    de_res$Gene  <- rownames(de_res)
    de_res$pct_1 <- rowMeans(counts[rownames(de_res), , drop = FALSE] > 0)
    write.csv(de_res, file = file.path(save_dir, paste0(out_prefix, ".csv")), row.names = FALSE)
  }
}

# run_de_blind: region-blind Dream LMM with per-group expression metrics
run_de_blind <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
  message(paste0("\n>>> [DREAM BLIND] STARTING FOR: ", dataset_name))
  form_de  <- ~ Treatment + log_depth + (1|sample_ID)
  geneExpr <- as.matrix(GetAssayData(seurat_obj, layer = "data"))
  counts   <- as.matrix(GetAssayData(seurat_obj, layer = "counts"))
  info     <- seurat_obj@meta.data
  tryCatch({
    dge  <- calcNormFactors(DGEList(counts[rowSums(counts) > 0, ]))
    vobj <- voomWithDreamWeights(dge, form_de, info, BPPARAM = param)
    fit  <- eBayes(dream(vobj, form_de, info, BPPARAM = param))
    target_coef <- grep("Treatment", colnames(fit$coefficients), value = TRUE)
    target_coef <- target_coef[length(target_coef)]
    if (length(target_coef) > 0) {
      de_res        <- topTable(fit, coef = target_coef, number = Inf, sort.by = "P", confint = TRUE)
      de_res$Gene   <- rownames(de_res)
      ref_level     <- levels(seurat_obj$Treatment)[1]
      target_level  <- levels(seurat_obj$Treatment)[2]
      cells_ref     <- which(seurat_obj$Treatment == ref_level)
      cells_target  <- which(seurat_obj$Treatment == target_level)
      de_res$pct_ref    <- rowMeans(geneExpr[rownames(de_res), cells_ref,    drop = FALSE] > 0)
      de_res$pct_target <- rowMeans(geneExpr[rownames(de_res), cells_target, drop = FALSE] > 0)
      de_res$avg_ref    <- rowMeans(geneExpr[rownames(de_res), cells_ref,    drop = FALSE])
      de_res$avg_target <- rowMeans(geneExpr[rownames(de_res), cells_target, drop = FALSE])
      write.csv(de_res, file = file.path(save_dir, paste0(out_prefix, ".csv")), row.names = FALSE)
      message(paste("    -> Saved:", file.path(save_dir, paste0(out_prefix, ".csv"))))
    }
  }, error = function(e) message(paste("    !! SKIPPING:", e$message)))
}

# run_pseudobulk_deseq2: pseudobulk DESeq2 validation
run_pseudobulk_deseq2 <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
  message(paste0("    >>> [PB] Running Pseudo-bulk DESeq2: ", dataset_name))
  cts <- AggregateExpression(seurat_obj, group.by = "sample_ID", assays = "RNA", slot = "counts")$RNA
  colnames(cts) <- gsub("-", "_", gsub("^g", "", colnames(cts)))
  colData <- seurat_obj@meta.data %>%
    dplyr::select(sample_ID, Treatment) %>%
    dplyr::distinct(sample_ID, .keep_all = TRUE)
  colData <- colData[match(colnames(cts), colData$sample_ID), ]
  rownames(colData) <- colData$sample_ID
  tryCatch({
    dds <- DESeqDataSetFromMatrix(countData = cts, colData = colData, design = ~ Treatment)
    dds <- dds[rowSums(counts(dds)) >= 10, ]
    dds <- DESeq(dds, quiet = TRUE)
    res <- as.data.frame(results(dds))
    res$Gene <- rownames(res)
    write.csv(res, file = file.path(save_dir, paste0("DESEQ2", out_prefix, ".csv")), row.names = FALSE)
    message(paste("    -> Saved PB:", file.path(save_dir, paste0("DESEQ2", out_prefix, ".csv"))))
  }, error = function(e) message(paste("    !! PB FAILED:", e$message)))
}

# prepare_metadata: use the configured contrast column (COMP$group_col) as the DE
# 'Treatment', restrict to the two contrast groups (ref_level + comparison_level,
# dropping any other level such as the FMT 'Cntrl' arm), scale sequencing depth,
# and relevel so ref_level is the reference. Restricting to exactly two levels also
# makes the region-blind / DESeq2 comparison-vs-reference logic unambiguous.
# comparison_level = NULL keeps every group (legacy behaviour).
prepare_metadata <- function(seurat_obj, group_col, ref_level, comparison_level = NULL) {
  names(seurat_obj@meta.data)[names(seurat_obj@meta.data) == group_col] <- "Treatment"
  if (!is.null(comparison_level) && nzchar(comparison_level)) {
    keep <- seurat_obj$Treatment %in% c(ref_level, comparison_level)
    if (!any(keep))
      stop(sprintf("prepare_metadata: no cells in contrast groups '%s' / '%s' (column '%s').",
                   ref_level, comparison_level, group_col))
    seurat_obj <- seurat_obj[, keep]
  }
  seurat_obj$Treatment <- droplevels(as.factor(seurat_obj$Treatment))
  if (ref_level %in% levels(seurat_obj$Treatment))
    seurat_obj$Treatment <- relevel(seurat_obj$Treatment, ref = ref_level)
  seurat_obj$log_depth <- as.numeric(scale(log10(seurat_obj$nFeature_RNA + 1)))
  return(seurat_obj)
}

In [ ]:
param <- SnowParam(workers = 5, type = "SOCK", progressbar = TRUE, exportglobals = FALSE)

base_dir <- cfg_paths$outputs$lmm_results_dir

# -- Iteration design (from config.yaml `composition`) -----------------------
#   DESeq2    -> deseq2_iterations per deviation magnitude
#   Dream LMM -> dream_iterations  per deviation magnitude (subset of the above)
group_col     <- COMP$group_col
ref_level     <- COMP$reference
comp_level    <- COMP$comparison   # non-reference contrast group; restricts the DE to
                                   # reference vs comparison (drops e.g. the FMT 'Cntrl' arm)
devs          <- unlist(COMP$deviations)
n_iter_deseq2 <- COMP$deseq2_iterations
n_iter_dream  <- COMP$dream_iterations
dev_lab <- function(d) format(d, trim = TRUE, scientific = FALSE)  # matches Python f"{dev:g}"
message("Contrast: ", comp_level, " vs ", ref_level, " (column ", group_col, ")")

# DESeq2 for every iteration; Dream (blind + region-aware) only when run_dream.
run_iter_suite <- function(obj, label, file_tag, dirs, run_dream) {
  run_pseudobulk_deseq2(obj,
    dataset_name = paste0(label, "_PB"),
    out_prefix   = paste0(label, "_", file_tag, "_PB"),
    save_dir     = dirs$pb)
  if (run_dream) {
    run_de_blind(obj,
      dataset_name = paste0(label, "_dream_blind"),
      out_prefix   = paste0(label, "_", file_tag, "_dream_blind"),
      save_dir     = dirs$local)
    run_de_dream(obj,
      dataset_name = paste0(label, "_dream_napari"),
      out_prefix   = paste0(label, "_", file_tag, "_dream_napari"),
      save_dir     = dirs$global,
      region_col   = "napari_region")
    run_de_dream(obj,
      dataset_name = paste0(label, "_dream_quint"),
      out_prefix   = paste0(label, "_", file_tag, "_dream_quint"),
      save_dir     = dirs$global,
      region_col   = "quint_region")
  }
}

target_cell_types <- c("Astrocytes", "Microglia")

run_ct_loop <- function(data_obj, dataset_label, dirs, run_dream) {
  for (ct in target_cell_types) {
    cell_col <- if (ct == "Astrocytes.cortex.hippocampus") "cell_type" else "ct_simple"
    obj_ct   <- data_obj[, data_obj@meta.data[[cell_col]] == ct]
    if (ncol(obj_ct) < 100) { message(paste("Skipping low cell count:", ct)); next }
    ct_clean <- gsub("[^A-Za-z0-9]", "_", ct)
    run_iter_suite(obj_ct,
      label     = paste0(dataset_label, "_", ct),
      file_tag  = ct_clean,
      dirs      = dirs,
      run_dream = run_dream)
    rm(obj_ct); gc()
  }
}

# -- Main loop: deviation magnitude x iteration x {UP, DOWN} ------------------
# Reconstruct the cortex-up/down subsets from the G1/G2 masks per iteration and
# write results into per-iteration subfolders so nothing overwrites.
for (dev in devs) {
  dlab <- dev_lab(dev)
  for (seed in 0:(n_iter_deseq2 - 1)) {
    g1_col <- paste0("G1_", seed, "_", dlab)   # cortex-up   (up_group high)
    g2_col <- paste0("G2_", seed, "_", dlab)   # cortex-down (mirror)
    if (!all(c(g1_col, g2_col) %in% colnames(full@meta.data))) {
      message(paste0("Skipping missing masks: ", g1_col, " / ", g2_col)); next
    }
    run_dream <- seed < n_iter_dream

    iter_root <- file.path(base_dir, paste0("dev_", dlab), paste0("seed_", seed))
    dirs <- list(
      global = file.path(iter_root, "Global_CT_Analysis"),
      local  = file.path(iter_root, "Local_Regional_Analysis"),
      pb     = file.path(iter_root, "Pseudobulk_Validation")
    )
    sapply(dirs, function(x) if (!dir.exists(x)) dir.create(x, recursive = TRUE))

    message(paste0("\n>>> dev=", dlab, " seed=", seed, "  (dream=", run_dream, ")"))
    up   <- prepare_metadata(full[, which(as.logical(full@meta.data[[g1_col]]))], group_col, ref_level, comp_level)
    down <- prepare_metadata(full[, which(as.logical(full@meta.data[[g2_col]]))], group_col, ref_level, comp_level)

    run_iter_suite(up,   label = "UP",   file_tag = "WHOLE", dirs = dirs, run_dream = run_dream)
    run_iter_suite(down, label = "DOWN", file_tag = "WHOLE", dirs = dirs, run_dream = run_dream)

    run_ct_loop(up,   "UP",   dirs, run_dream)
    run_ct_loop(down, "DOWN", dirs, run_dream)

    rm(up, down); gc()
  }
}

message("\n--- ALL ANALYSES COMPLETE ---")